# SmolVLA Optuna Hyperparameter Optimization (Local RTX 3050)

Run Optuna HPO locally on RTX 3050 (4GB VRAM) to find optimal hyperparameters for SmolVLA.

- **Dataset:** `AdithyaRajendran/so101_grab_brain_t2` (241 episodes, 100832 frames)
- **Task:** *"Grab the grey brain toy and place it inside the green container"*
- **Constraint:** batch_size=1, train_expert_only=True, freeze_vision_encoder=True (4GB VRAM)
- **Trials:** 100, 3500 steps each (~24 min/trial incl. model load)
- **Total runtime:** ~28-32 hours (with pruning saving ~20-30%)
- **Metric:** Average loss over final 1000 steps

### Timing (measured on RTX 3050):
- Model load: ~65s per trial
- Training: ~0.4s/step → 3500 steps = ~23 min
- Total per trial: ~24 min
- Without pruning: 100 × 24 min = ~40 hrs
- With pruning (~40-50% of trials pruned at ~50%): **~28-32 hrs**

### Previous models (all used identical hyperparams except chunk_size/n_action_steps):
- lr=1e-4, weight_decay=1e-10, freeze_vision=True always
- Best was ~33% success. Massive unexplored search space.

In [ ]:
# Install dependencies (run once)
!pip install -q optuna matplotlib

In [ ]:
import subprocess
import re
import shutil
import json
import time
from pathlib import Path

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

In [ ]:
# === CONFIGURATION ===

LEROBOT_DIR = Path("/home/adithya/lerobot")
TRAIN_SCRIPT = LEROBOT_DIR / "src" / "lerobot" / "scripts" / "lerobot_train.py"
OUTPUT_BASE = LEROBOT_DIR / "outputs" / "optuna_hpo"
DB_PATH = OUTPUT_BASE / "optuna_smolvla.db"
STUDY_NAME = "smolvla_hpo_v2"

DATASET_REPO_ID = "AdithyaRajendran/so101_grab_brain_t2"
POLICY_PATH = "lerobot/smolvla_base"

# Fixed params (GPU constraint)
BATCH_SIZE = 1

# --- SETTINGS FOR 100-TRIAL RUN ---
# Measured: ~0.4s/step + ~65s model load per trial
# 3500 steps × 0.4s + 65s = ~24 min/trial
# 100 trials × 24 min = ~40 hrs max (with pruning: ~28-32 hrs)
STEPS_PER_TRIAL = 3500
N_TRIALS = 100
LOG_FREQ = 100  # 35 log entries per trial

# Fixed augmentation params
SHARPNESS_RANGE = [0.5, 1.5]
HUE_RANGE = [-0.05, 0.05]
SATURATION_RANGE = [0.5, 1.5]

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
print(f"Study DB: {DB_PATH}")
print(f"Output base: {OUTPUT_BASE}")
print(f"Steps/trial: {STEPS_PER_TRIAL:,}")
print(f"Trials: {N_TRIALS}")
est_max = N_TRIALS * (STEPS_PER_TRIAL * 0.4 + 65) / 3600
print(f"Estimated runtime: ~{est_max:.0f} hrs max, ~{est_max * 0.75:.0f} hrs with pruning")

In [ ]:
def build_train_command(trial_params, output_dir):
    """Build the lerobot_train.py CLI command from trial parameters."""
    p = trial_params

    norm_map = json.dumps({
        "ACTION": "MEAN_STD",
        "STATE": "MEAN_STD",
        "VISUAL": "IDENTITY",
    })

    rename_map = json.dumps({
        "observation.images.front": "observation.images.camera1",
        "observation.images.wrist": "observation.images.camera2",
    })

    affine_degrees = p["affine_degrees"]
    affine_translate = p["affine_translate"]
    brightness = p["color_jitter_brightness"]
    contrast = p["color_jitter_contrast"]

    # Build image transforms as a single JSON dict
    # (draccus cannot parse nested --dataset.image_transforms.tfs.X.Y.Z args)
    tfs_dict = {
        "brightness": {
            "type": "ColorJitter",
            "kwargs": {"brightness": [round(1.0 - brightness, 3), round(1.0 + brightness, 3)]}
        },
        "contrast": {
            "type": "ColorJitter",
            "kwargs": {"contrast": [round(1.0 - contrast, 3), round(1.0 + contrast, 3)]}
        },
        "saturation": {
            "type": "ColorJitter",
            "kwargs": {"saturation": [SATURATION_RANGE[0], SATURATION_RANGE[1]]}
        },
        "hue": {
            "type": "ColorJitter",
            "kwargs": {"hue": [HUE_RANGE[0], HUE_RANGE[1]]}
        },
        "sharpness": {
            "type": "SharpnessJitter",
            "kwargs": {"sharpness": [SHARPNESS_RANGE[0], SHARPNESS_RANGE[1]]}
        },
        "affine": {
            "type": "RandomAffine",
            "kwargs": {
                "degrees": [round(-affine_degrees, 2), round(affine_degrees, 2)],
                "translate": [round(affine_translate, 4), round(affine_translate, 4)]
            }
        },
    }

    cmd = [
        "python", str(TRAIN_SCRIPT),
        f"--dataset.repo_id={DATASET_REPO_ID}",
        f"--output_dir={output_dir}",
        f"--job_name=optuna_trial",
        f"--policy.path={POLICY_PATH}",
        f"--policy.push_to_hub=false",
        f"--policy.chunk_size={p['chunk_size']}",
        f"--policy.n_action_steps={p['n_action_steps']}",
        f"--policy.optimizer_lr={p['learning_rate']}",
        f"--policy.optimizer_weight_decay={p['weight_decay']}",
        f"--policy.train_expert_only=true",
        f"--policy.freeze_vision_encoder=true",
        f"--policy.scheduler_warmup_steps=200",
        f"--policy.scheduler_decay_steps={STEPS_PER_TRIAL}",
        f"--policy.scheduler_decay_lr={p['learning_rate'] * 0.025}",
        f"--batch_size={BATCH_SIZE}",
        f"--steps={STEPS_PER_TRIAL}",
        f"--log_freq={LOG_FREQ}",
        f"--save_checkpoint=false",
        f"--eval_freq=0",
        f"--num_workers=2",
        f"--policy.normalization_mapping={norm_map}",
        f"--rename_map={rename_map}",
        # Image transforms - pass tfs as a single JSON dict
        f"--dataset.image_transforms.enable=true",
        f"--dataset.image_transforms.max_num_transforms=3",
        f"--dataset.image_transforms.random_order=false",
        f"--dataset.image_transforms.tfs={json.dumps(tfs_dict)}",
    ]

    return cmd

In [ ]:
def parse_loss_from_output(line):
    """Parse loss value from a training log line.

    Actual format: INFO 2026-03-02 19:39:55 ot_train.py:501 step:10 smpl:10 ... loss:0.274 ...
    """
    match = re.search(r'loss:(\d+\.\d+)', line)
    if match:
        return float(match.group(1))
    return None


def parse_step_from_output(line):
    """Parse step number from a training log line.

    Handles: step:10, step:100, step:1k, step:3.5k, etc.
    """
    match = re.search(r'step:(\d+\.?\d*)(k?)', line)
    if match:
        val = float(match.group(1))
        if match.group(2) == 'k':
            val *= 1000
        return int(val)
    return None


def run_trial(trial_params, trial_number, trial_obj=None):
    """Run a single training trial and return the average loss over the final 1000 steps."""
    output_dir = OUTPUT_BASE / f"trial_{trial_number:03d}"

    # Clean up any previous run
    if output_dir.exists():
        shutil.rmtree(output_dir)

    cmd = build_train_command(trial_params, str(output_dir))
    print(f"\n{'='*60}")
    print(f"Trial {trial_number} | Started at {time.strftime('%H:%M:%S')}")
    print(f"Params: {json.dumps({k: round(v, 6) if isinstance(v, float) else v for k, v in trial_params.items()}, indent=2)}")
    print(f"{'='*60}")

    trial_start = time.time()
    losses = []  # (step, loss) pairs
    all_output = []  # Capture ALL output for error diagnosis
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=str(LEROBOT_DIR),
    )

    for line in process.stdout:
        line = line.strip()
        all_output.append(line)
        loss = parse_loss_from_output(line)
        step = parse_step_from_output(line)

        if loss is not None and step is not None:
            losses.append((step, loss))
            elapsed = time.time() - trial_start
            print(f"  step={step:>5d}/{STEPS_PER_TRIAL}  loss={loss:.4f}  [{elapsed:.0f}s elapsed]")

            # Report to Optuna for pruning
            if trial_obj is not None:
                trial_obj.report(loss, step)
                if trial_obj.should_prune():
                    process.terminate()
                    process.wait()
                    if output_dir.exists():
                        shutil.rmtree(output_dir)
                    print(f"  PRUNED at step {step} after {elapsed:.0f}s")
                    raise optuna.TrialPruned(f"Pruned at step {step}")

    retcode = process.wait()
    trial_elapsed = time.time() - trial_start

    # Clean up output dir to save disk space
    if output_dir.exists():
        shutil.rmtree(output_dir)

    if not losses:
        print(f"\nERROR: No losses captured for trial {trial_number} (exit code {retcode}, {trial_elapsed:.0f}s)")
        print(f"--- FULL SUBPROCESS OUTPUT (last 50 lines) ---")
        for line in all_output[-50:]:
            print(f"  {line}")
        print(f"--- END OUTPUT ---")
        return float('inf')

    # Average loss over final 1000 steps (last 10 log entries with log_freq=100)
    final_losses = [l for s, l in losses if s > STEPS_PER_TRIAL - 1000]
    if not final_losses:
        # Fallback: use last 5 entries
        final_losses = [l for _, l in losses[-5:]]

    avg_loss = sum(final_losses) / len(final_losses)
    print(f"\nTrial {trial_number} done in {trial_elapsed/60:.1f} min | avg final loss: {avg_loss:.5f} (from {len(final_losses)} entries)")
    return avg_loss

In [ ]:
def objective(trial):
    """Optuna objective function."""

    # Hyperparameter search space
    learning_rate = trial.suggest_float("learning_rate", 5e-6, 5e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-10, 1e-3, log=True)
    chunk_size = trial.suggest_categorical("chunk_size", [10, 20, 30, 50])

    # n_action_steps as a ratio of chunk_size
    action_ratio = trial.suggest_categorical("action_ratio", [0.5, 1.0])
    n_action_steps = max(1, int(chunk_size * action_ratio))

    # Augmentation strengths
    affine_degrees = trial.suggest_float("affine_degrees", 3.0, 15.0)
    affine_translate = trial.suggest_float("affine_translate", 0.03, 0.15)
    color_jitter_brightness = trial.suggest_float("color_jitter_brightness", 0.1, 0.4)
    color_jitter_contrast = trial.suggest_float("color_jitter_contrast", 0.1, 0.4)

    trial_params = {
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
        "chunk_size": chunk_size,
        "n_action_steps": n_action_steps,
        "affine_degrees": affine_degrees,
        "affine_translate": affine_translate,
        "color_jitter_brightness": color_jitter_brightness,
        "color_jitter_contrast": color_jitter_contrast,
    }

    avg_loss = run_trial(trial_params, trial.number, trial_obj=trial)
    return avg_loss

In [ ]:
# === CREATE STUDY WITH SEEDED TRIALS ===

storage = f"sqlite:///{DB_PATH}"

# Delete existing study if re-running from scratch
try:
    optuna.delete_study(study_name=STUDY_NAME, storage=storage)
    print(f"Deleted existing study '{STUDY_NAME}'")
except KeyError:
    pass

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=storage,
    direction="minimize",
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(
        n_startup_trials=5,    # Don't prune first 5 trials (need baselines)
        n_warmup_steps=1000,   # Don't prune before step 1000 (let loss stabilize)
        interval_steps=100,    # Check pruning every 100 steps (aligned with LOG_FREQ)
    ),
    load_if_exists=False,
)

# Seed trial 0: v5 baseline (lr=1e-4, chunk=10, minimal augmentation)
study.enqueue_trial({
    "learning_rate": 1e-4,
    "weight_decay": 1e-10,
    "chunk_size": 10,
    "action_ratio": 1.0,
    "affine_degrees": 3.0,
    "affine_translate": 0.03,
    "color_jitter_brightness": 0.1,
    "color_jitter_contrast": 0.1,
})

# Seed trial 1: New hypothesis (lower lr, larger chunk, strong augmentation)
study.enqueue_trial({
    "learning_rate": 2.5e-5,
    "weight_decay": 1e-4,
    "chunk_size": 30,
    "action_ratio": 0.5,  # n_action_steps = 15
    "affine_degrees": 10.0,
    "affine_translate": 0.1,
    "color_jitter_brightness": 0.3,
    "color_jitter_contrast": 0.3,
})

# Seed trial 2: Medium lr, chunk=20, moderate augmentation
study.enqueue_trial({
    "learning_rate": 5e-5,
    "weight_decay": 1e-6,
    "chunk_size": 20,
    "action_ratio": 1.0,
    "affine_degrees": 7.0,
    "affine_translate": 0.07,
    "color_jitter_brightness": 0.2,
    "color_jitter_contrast": 0.2,
})

print(f"Study created: {STUDY_NAME}")
print(f"Storage: {storage}")
print(f"3 seeded trials enqueued + {N_TRIALS - 3} TPE trials")
print(f"Pruner: MedianPruner(n_startup_trials=5, n_warmup_steps=1000, interval_steps=100)")
print(f"Estimated total runtime: ~28-32 hrs with pruning")

In [ ]:
# === RUN OPTIMIZATION ===
# 100 trials × ~24 min each = ~40 hrs max, ~28-32 hrs with pruning
# Progress is saved to SQLite — you can interrupt and resume with the cell at the bottom.

print(f"Starting optimization: {N_TRIALS} trials x {STEPS_PER_TRIAL:,} steps")
print(f"Start time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
est_max = N_TRIALS * (STEPS_PER_TRIAL * 0.4 + 65) / 3600
print(f"Estimated end: ~{est_max * 0.75:.0f}-{est_max:.0f} hrs from now")
print()

overall_start = time.time()
study.optimize(objective, n_trials=N_TRIALS)
overall_elapsed = time.time() - overall_start

print(f"\nOptimization complete!")
print(f"Total time: {overall_elapsed/3600:.1f} hours")
print(f"Completed trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])}")
print(f"Pruned trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])}")

In [ ]:
# === RESULTS ===

print("\n" + "=" * 70)
print("OPTUNA HPO RESULTS")
print("=" * 70)

completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
print(f"\nCompleted: {len(completed)} | Pruned: {len(pruned)} | Total: {len(study.trials)}")

print(f"\nBest trial: #{study.best_trial.number}")
print(f"Best value (avg final loss): {study.best_trial.value:.5f}")
print(f"\nBest hyperparameters:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")

best = study.best_trial.params
best_n_action_steps = max(1, int(best["chunk_size"] * best["action_ratio"]))
print(f"  n_action_steps (derived): {best_n_action_steps}")

print(f"\nAll trials (sorted by loss):")
print(f"{'#':>3} {'Value':>10} {'State':>10} | {'lr':>10} {'wd':>10} {'chunk':>5} {'ratio':>5} {'aff_deg':>7} {'aff_tr':>6} {'bright':>6} {'contr':>6}")
print("-" * 105)
sorted_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else float('inf'))
for trial in sorted_trials:
    p = trial.params
    val = f"{trial.value:.5f}" if trial.value is not None else "N/A"
    print(
        f"{trial.number:>3} {val:>10} {trial.state.name:>10} | "
        f"{p.get('learning_rate', 0):>10.2e} {p.get('weight_decay', 0):>10.2e} "
        f"{p.get('chunk_size', 0):>5} {p.get('action_ratio', 0):>5.1f} "
        f"{p.get('affine_degrees', 0):>7.1f} {p.get('affine_translate', 0):>6.3f} "
        f"{p.get('color_jitter_brightness', 0):>6.2f} {p.get('color_jitter_contrast', 0):>6.2f}"
    )

In [ ]:
# === GENERATE FINAL TRAINING COMMAND ===

best = study.best_trial.params
best_n_action_steps = max(1, int(best["chunk_size"] * best["action_ratio"]))

print("\n" + "=" * 70)
print("BEST PARAMS FOR FINAL TRAINING (copy to smolvla_final_training.ipynb)")
print("=" * 70)
print(f"""
# --- Paste into smolvla_final_training.ipynb config cell ---
LEARNING_RATE = {best['learning_rate']}
WEIGHT_DECAY = {best['weight_decay']}
CHUNK_SIZE = {best['chunk_size']}
N_ACTION_STEPS = {best_n_action_steps}
AFFINE_DEGREES = {best['affine_degrees']}
AFFINE_TRANSLATE = {best['affine_translate']}
COLOR_JITTER_BRIGHTNESS = {best['color_jitter_brightness']}
COLOR_JITTER_CONTRAST = {best['color_jitter_contrast']}
""")

In [ ]:
# === VISUALIZATION (matplotlib — no nbformat needed) ===

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

if len(completed_trials) >= 3:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"Optuna HPO Results - {len(completed_trials)} completed trials", fontsize=14)

    # 1. Optimization history
    ax = axes[0, 0]
    trial_nums = [t.number for t in completed_trials]
    trial_vals = [t.value for t in completed_trials if t.value != float('inf')]
    trial_nums_valid = [t.number for t in completed_trials if t.value != float('inf')]
    if trial_vals:
        best_so_far = [min(trial_vals[:i+1]) for i in range(len(trial_vals))]
        ax.scatter(trial_nums_valid, trial_vals, alpha=0.6, label='Trial loss')
        ax.plot(trial_nums_valid, best_so_far, 'r-', linewidth=2, label='Best so far')
    ax.set_xlabel('Trial #')
    ax.set_ylabel('Avg final loss')
    ax.set_title('Optimization History')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 2. Learning rate vs loss
    ax = axes[0, 1]
    valid_trials = [t for t in completed_trials if t.value != float('inf')]
    if valid_trials:
        lrs = [t.params['learning_rate'] for t in valid_trials]
        vals = [t.value for t in valid_trials]
        ax.scatter(lrs, vals, alpha=0.6)
        ax.set_xscale('log')
    ax.set_xlabel('Learning Rate')
    ax.set_ylabel('Avg final loss')
    ax.set_title('Learning Rate vs Loss')
    ax.grid(True, alpha=0.3)

    # 3. Chunk size distribution
    ax = axes[1, 0]
    chunk_sizes = [10, 20, 30, 50]
    chunk_data = []
    chunk_labels = []
    for cs in chunk_sizes:
        cs_vals = [t.value for t in valid_trials if t.params['chunk_size'] == cs]
        if cs_vals:
            chunk_data.append(cs_vals)
            chunk_labels.append(str(cs))
    if chunk_data:
        ax.boxplot(chunk_data, labels=chunk_labels)
    ax.set_xlabel('Chunk Size')
    ax.set_ylabel('Avg final loss')
    ax.set_title('Loss by Chunk Size')
    ax.grid(True, alpha=0.3)

    # 4. Weight decay vs loss
    ax = axes[1, 1]
    if valid_trials:
        wds = [t.params['weight_decay'] for t in valid_trials]
        vals = [t.value for t in valid_trials]
        ax.scatter(wds, vals, alpha=0.6)
        ax.set_xscale('log')
    ax.set_xlabel('Weight Decay')
    ax.set_ylabel('Avg final loss')
    ax.set_title('Weight Decay vs Loss')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = OUTPUT_BASE / "optuna_results.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"Plots saved to: {plot_path}")
    plt.show()
else:
    print(f"Only {len(completed_trials)} completed trials - need at least 3 for plots.")

In [ ]:
# === PARAM IMPORTANCE (needs enough completed trials) ===

if len([t for t in completed_trials if t.value != float('inf')]) >= 8:
    try:
        importances = optuna.importance.get_param_importances(study)
        print("\nParameter Importances:")
        print("-" * 40)
        for param, imp in sorted(importances.items(), key=lambda x: -x[1]):
            bar = '#' * int(imp * 50)
            print(f"  {param:>25s}: {imp:.3f} {bar}")

        fig, ax = plt.subplots(figsize=(10, 5))
        params = list(importances.keys())
        values = list(importances.values())
        sorted_idx = np.argsort(values)
        ax.barh([params[i] for i in sorted_idx], [values[i] for i in sorted_idx])
        ax.set_xlabel('Importance')
        ax.set_title('Hyperparameter Importance')
        plt.tight_layout()
        imp_path = OUTPUT_BASE / "param_importance.png"
        plt.savefig(imp_path, dpi=150, bbox_inches='tight')
        print(f"\nImportance plot saved to: {imp_path}")
        plt.show()
    except Exception as e:
        print(f"Could not compute importance: {e}")
else:
    valid = len([t for t in completed_trials if t.value != float('inf')])
    print(f"Need at least 8 valid completed trials for importance analysis (have {valid})")

In [ ]:
# === RESUME AN INTERRUPTED STUDY ===
# If the kernel was interrupted, run this cell INSTEAD OF the "CREATE STUDY" cell,
# then run the "RUN OPTIMIZATION" cell — it will continue where it left off.

# study = optuna.load_study(
#     study_name=STUDY_NAME,
#     storage=f"sqlite:///{DB_PATH}",
# )
# completed = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
# pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
# remaining = N_TRIALS - len(study.trials)
# print(f"Loaded study: {completed} completed, {pruned} pruned, {remaining} remaining")
# if remaining > 0:
#     study.optimize(objective, n_trials=remaining)
# else:
#     print("All trials already completed!")